# Task 2 — Near-Duplicate Document Detection at Scale

## HOW TO RUN (read first)
1. Click **Connect** (top right) and wait for the green tick. Then run **Cell 1**. It downloads your data folder from Google Drive automatically (no mounting, no paths to type).
2. If Cell 1 says files are missing, you must first run your dataset-builder notebook (the one that prints `[d] 112571 docs`) until it finishes, then run Cell 1 again.
3. Then press **Runtime -> Run all** (or Shift+Enter on each cell, top to bottom).
4. When Section 1 finishes you will have `submission.csv` in the Colab file panel (folder icon on the left). Download it.

## What the method does (one paragraph)
Comparing every document with every other one is too slow. Instead: (1) turn each document into a set of 5-word shingles; (2) shrink each set to a short MinHash signature; (3) LSH banding puts documents with similar signatures into the same bucket, so only those few pairs become candidates; (4) every candidate is checked with the **exact** Jaccard formula. Precision is therefore 1 by construction; recall is controlled by the band settings (B=21 bands, R=6 rows).

In [ ]:
# ---------------- CELL 1: imports + automatic data download/loading ----------------
import os, time, math, statistics, gc, itertools, glob, subprocess, sys
from collections import defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import sparse

FOLDER_ID = "1VJycVGjec7N9rSRSdid8_wX36DtHgK5K"   # your Google Drive folder (must be shared: Anyone with the link)
NEEDED = ["docs.csv", "dev_docs.csv", "dev_ground_truth.csv"]

def find_file(name):
    for root in ["/content/data", "/content", "."]:
        hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
        if hits:
            return hits[0]
    return None

paths = {n: find_file(n) for n in NEEDED}
if any(p is None for p in paths.values()):
    print("Files not on this runtime yet -> downloading folder from Google Drive ...")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gdown"], check=True)
    import gdown
    gdown.download_folder(id=FOLDER_ID, output="/content/data", quiet=False, use_cookies=False)
    paths = {n: find_file(n) for n in NEEDED}

print(paths)
missing = [n for n, p in paths.items() if p is None]
if missing:
    raise FileNotFoundError(
        f"Still missing {missing}.\n"
        "FIX 1: in Google Drive, right-click the folder -> Share -> General access -> 'Anyone with the link' (Viewer), then re-run.\n"
        "FIX 2: download the 3 CSV files to your computer and drag them into the Colab file panel (folder icon on the left), then re-run.")

DATA_DIR = os.path.dirname(paths["docs.csv"])
print("Using DATA_DIR =", DATA_DIR)

docs  = pd.read_csv(paths["docs.csv"],     dtype=str, keep_default_na=False)
dev   = pd.read_csv(paths["dev_docs.csv"], dtype=str, keep_default_na=False)
truth = pd.read_csv(paths["dev_ground_truth.csv"], dtype=str)
print(f"docs={len(docs)}  dev={len(dev)}  dev_truth_pairs={len(truth)}")
assert len(docs) > 100000, "docs.csv looks too small - is this the right file?"

## 0. Core implementation (all stages, modular)

In [ ]:
import time, itertools
import numpy as np

U64 = np.uint64
MASK64 = (1 << 64) - 1

# ----------------------------------------------------------------------------
# Tunable parameters (justified in the notebook)
# ----------------------------------------------------------------------------
PARAMS = dict(
    K=126, R=6, B=21,      # MinHash size, rows per band, bands  (K = R*B)
    SEED=12345,
    USE_DF=True,           # drop corpus-wide boilerplate shingles from the *sketch* only
    DF_CAP=None,           # None -> max(DF_MIN, DF_FRAC * n_distinct_docs)
    DF_FRAC=0.002, DF_MIN=30,
    MAX_DROP=0.5,          # if a doc would lose >50% of its shingles, sketch the full set
    SHORT_M=20,            # docs with < SHORT_M shingles are always sketched in full
    MAX_BUCKET=400,        # safety valve for absurdly large LSH buckets
    BLOCK=4096,            # shingles per MinHash block (bounds temp memory)
)
assert PARAMS["K"] == PARAMS["R"] * PARAMS["B"]

_MULT = [U64(c) for c in (0x9E3779B97F4A7C15, 0xC2B2AE3D27D4EB4F, 0x165667B19E3779F9,
                          0xD6E8FEB86659FD93, 0xFF51AFD7ED558CCD)]
_GOLD = U64(0x9E3779B97F4A7C15)


def _mix(x):
    """splitmix64 finaliser, vectorised on uint64 arrays (wrap-around arithmetic)."""
    x = x ^ (x >> U64(30)); x = x * U64(0xBF58476D1CE4E5B9)
    x = x ^ (x >> U64(27)); x = x * U64(0x94D049BB133111EB)
    return x ^ (x >> U64(31))


# ----------------------------------------------------------------------------
# Stage 1: shingling + exact-duplicate collapsing
# ----------------------------------------------------------------------------
def build_shingle_index(texts):
    """Each doc -> sorted array of unique 64-bit hashes of its word 5-grams
    (docs with <5 words -> one hash of the whole word tuple).
    Docs with an identical shingle set are collapsed onto one representative.
    Returns CSR arrays (U, offs, lens) over representatives and `members`
    (list of doc-index lists, members[r][0] is the representative)."""
    groups, chunks = {}, []
    for i, t in enumerate(texts):
        toks = t.split()
        m = len(toks)
        if m < 5:
            h = np.array([hash(tuple(toks)) & MASK64], dtype=np.uint64)
        else:
            th = _mix(np.fromiter(map(hash, toks), dtype=np.int64, count=m).view(np.uint64))
            g = th[:m - 4] * _MULT[0]
            for j in range(1, 5):                       # position-dependent combine
                g += th[j:j + m - 4] * _MULT[j]
            h = np.unique(_mix(g))
        fp = (h.size, int(h.sum()), int(np.bitwise_xor.reduce(h)))
        lst = groups.get(fp)
        if lst is None:
            groups[fp] = [i]
            chunks.append(h)
        else:
            lst.append(i)
    members = list(groups.values())
    lens = np.fromiter((c.size for c in chunks), dtype=np.int64, count=len(chunks))
    offs = np.zeros(len(chunks) + 1, dtype=np.int64)
    np.cumsum(lens, out=offs[1:])
    U = np.concatenate(chunks) if chunks else np.empty(0, np.uint64)
    return U, offs, lens, members


# ----------------------------------------------------------------------------
# Stage 2: boilerplate detection (document frequency of shingles)
# ----------------------------------------------------------------------------
def boilerplate_mask(U, offs, lens, p):
    """Returns (is_common, use_full, info). Shingles occurring in > cap docs are
    'boilerplate' and are excluded from the MinHash sketch (never from verification)."""
    nr = lens.size
    cap = p["DF_CAP"] or max(p["DF_MIN"], int(p["DF_FRAC"] * nr))
    info = dict(cap=cap, n_common=0, frac_common=0.0, n_full_docs=0)
    if not p["USE_DF"] or nr == 0:
        return None, None, info
    uniq, cnt = np.unique(U, return_counts=True)
    common = uniq[cnt > cap]
    del uniq, cnt
    info["n_common"] = int(common.size)
    if common.size == 0:
        return None, None, info
    pos = np.searchsorted(common, U)
    pos[pos == common.size] = common.size - 1
    is_common = common[pos] == U
    del pos
    ncom = np.add.reduceat(is_common, offs[:-1], dtype=np.int64)
    kept = lens - ncom
    use_full = (lens < p["SHORT_M"]) | (kept < (1 - p["MAX_DROP"]) * lens)
    info["frac_common"] = float(is_common.mean())
    info["n_full_docs"] = int(use_full.sum())
    return is_common, use_full, info


# ----------------------------------------------------------------------------
# Stage 3: MinHash signatures (K hash functions h(x) = (a*x + b mod 2^64) >> 32)
# ----------------------------------------------------------------------------
def minhash_signatures(U, offs, lens, is_common, use_full, p):
    nr, K, BLOCK = lens.size, p["K"], p["BLOCK"]
    rng = np.random.default_rng(p["SEED"])
    A = rng.integers(1, 1 << 63, size=K, dtype=np.uint64) | U64(1)
    Bc = rng.integers(0, 1 << 63, size=K, dtype=np.uint64)
    sig = np.empty((nr, K), dtype=np.uint32)
    for i in range(nr):
        s, e = offs[i], offs[i + 1]
        x = U[s:e]
        if is_common is not None and not use_full[i]:
            x = x[~is_common[s:e]]
        best = None
        for q in range(0, x.size, BLOCK):
            h = (x[q:q + BLOCK, None] * A[None, :] + Bc[None, :]) >> U64(32)
            mn = h.min(axis=0)
            best = mn if best is None else np.minimum(best, mn)
        sig[i] = best
    return sig


# ----------------------------------------------------------------------------
# Stage 4: LSH banding -> candidate pairs (packed as i*nr + j, i<j)
# ----------------------------------------------------------------------------
def lsh_candidates(sig, p):
    nr = sig.shape[0]
    R, B, maxb = p["R"], p["B"], p["MAX_BUCKET"]
    out, oversize = [], 0
    for b in range(B):
        key = np.zeros(nr, dtype=np.uint64)
        for j in range(R):
            key = _mix(key + sig[:, b * R + j].astype(np.uint64) + _GOLD)
        order = np.argsort(key, kind="stable")
        ks = key[order]
        brk = np.flatnonzero(ks[1:] != ks[:-1]) + 1
        starts = np.concatenate(([0], brk)); ends = np.concatenate((brk, [nr]))
        sz = ends - starts
        m2 = sz == 2                                     # the overwhelmingly common case
        a, c = order[starts[m2]], order[starts[m2] + 1]
        out.append(np.minimum(a, c).astype(np.int64) * nr + np.maximum(a, c))
        for s, e in zip(starts[sz > 2], ends[sz > 2]):   # rare bigger buckets
            if e - s > maxb:
                oversize += 1
                continue
            mem = np.sort(order[s:e]).astype(np.int64)
            i, j = np.triu_indices(mem.size, 1)
            out.append(mem[i] * nr + mem[j])
    cand = np.unique(np.concatenate(out)) if out else np.empty(0, np.int64)
    return cand, oversize


# ----------------------------------------------------------------------------
# Stage 5: exact verification
# ----------------------------------------------------------------------------
def _inter_size(a, b):
    if a.size > b.size:
        a, b = b, a
    idx = np.searchsorted(b, a)
    idx[idx == b.size] = b.size - 1
    return int(np.count_nonzero(b[idx] == a))


def verify_pairs(cand, nr, U, offs, lens, t):
    ii, jj = cand // nr, cand % nr
    li, lj = lens[ii], lens[jj]
    ok = np.minimum(li, lj) >= t * np.maximum(li, lj) - 1e-12   # J <= min/max (size filter)
    J = np.full(cand.size, -1.0)
    for k in np.flatnonzero(ok):
        i, j = ii[k], jj[k]
        inter = _inter_size(U[offs[i]:offs[i + 1]], U[offs[j]:offs[j + 1]])
        J[k] = inter / (li[k] + lj[k] - inter)
    return ii, jj, J


# ----------------------------------------------------------------------------
# Driver
# ----------------------------------------------------------------------------
def _find(ids, texts, threshold=0.8, params=None, keep_internal=False):
    p = dict(PARAMS); p.update(params or {})
    assert p["K"] == p["R"] * p["B"]
    T, tm = time.perf_counter, {}

    t0 = T(); U, offs, lens, members = build_shingle_index(texts); tm["1 shingle+dedupe"] = T() - t0
    nr = lens.size
    t0 = T(); is_common, use_full, dfinfo = boilerplate_mask(U, offs, lens, p); tm["2 boilerplate df"] = T() - t0
    t0 = T(); sig = minhash_signatures(U, offs, lens, is_common, use_full, p); tm["3 minhash"] = T() - t0
    del is_common
    t0 = T(); cand, oversize = lsh_candidates(sig, p); tm["4 lsh"] = T() - t0
    t0 = T(); ii, jj, J = verify_pairs(cand, nr, U, offs, lens, threshold); tm["5 verify"] = T() - t0

    t0 = T()
    pairs = set()
    def add(x, y):
        if x != y:
            pairs.add((x, y) if x < y else (y, x))
    for g in members:                                   # exact-duplicate groups: J = 1
        if len(g) > 1:
            for a, b in itertools.combinations(g, 2):
                add(ids[a], ids[b])
    for k in np.flatnonzero(J >= threshold):            # near-duplicate groups
        for a in members[ii[k]]:
            for b in members[jj[k]]:
                add(ids[a], ids[b])
    tm["6 expand"] = T() - t0

    diag = dict(times=tm, total=sum(tm.values()), n_docs=len(texts), n_reps=nr,
                n_candidates=int(cand.size), n_size_ok=int((J >= 0).sum()),
                n_accepted_rep_pairs=int((J >= threshold).sum()),
                oversize_buckets=oversize, df=dfinfo, params=p,
                n_shingles=int(U.size))
    if keep_internal:
        diag.update(U=U, offs=offs, lens=lens, members=members, cand=cand,
                    cand_J=J, sig=sig)
    return pairs, diag


LAST_DIAG = {}
def find_similar_pairs(ids, texts, threshold=0.8):
    """ids: list[str], texts: list[str]. Returns set of (id1, id2), id1 < id2,
    with Jaccard(word-5-gram sets) >= threshold."""
    global LAST_DIAG
    pairs, LAST_DIAG = _find(list(ids), list(texts), threshold)
    return pairs

## 1. Working pipeline on the full corpus + `submission.csv`

In [ ]:
ids_full, texts_full = docs["id"].tolist(), docs["text"].tolist()
t0 = time.perf_counter()
pairs_full = find_similar_pairs(ids_full, texts_full)
full_runtime = time.perf_counter() - t0
d = LAST_DIAG
print(f"TOTAL runtime: {full_runtime:.1f}s  ({full_runtime/60:.2f} min)   pairs found: {len(pairs_full)}")
print("stage times:", {k: round(v,1) for k,v in d["times"].items()})
print({k: d[k] for k in ["n_docs","n_reps","n_shingles","n_candidates","n_size_ok","n_accepted_rep_pairs","oversize_buckets"]}, d["df"])

sub = pd.DataFrame(sorted(pairs_full), columns=["id1","id2"])
assert (sub.id1 < sub.id2).all() and not sub.duplicated().any()
sub.to_csv("submission.csv", index=False)
sub.head()

## 2. Correctness report on the dev set

In [ ]:
def shingle_set(text):                       # independent reference implementation
    w = text.split()
    return {tuple(w)} if len(w) < 5 else {tuple(w[i:i+5]) for i in range(len(w)-4)}
def exact_jaccard(a, b):
    A, B = shingle_set(a), shingle_set(b); return len(A & B) / len(A | B)

def prf(pred, gold):
    tp = len(pred & gold); p = tp/len(pred) if pred else 1.0; r = tp/len(gold) if gold else 1.0
    return p, r, (2*p*r/(p+r) if p+r else 0.0), tp

gold = {tuple(sorted(x)) for x in zip(truth.id1, truth.id2)}
truth_J = {tuple(sorted((a,b))): float(j) for a,b,j in zip(truth.id1, truth.id2, truth.jaccard)}
dev_ids, dev_texts = dev["id"].tolist(), dev["text"].tolist()
dev_text = dict(zip(dev_ids, dev_texts))

t0 = time.perf_counter()
dev_pairs, dd = _find(dev_ids, dev_texts, keep_internal=True)
print(f"dev runtime {time.perf_counter()-t0:.1f}s")
P, Rc, F1, tp = prf(dev_pairs, gold)
print(f"Precision={P:.4f}  Recall={Rc:.4f}  F1={F1:.4f}   (TP={tp}, found={len(dev_pairs)}, truth={len(gold)})")

# theoretical probability a pair is examined
def p_examined(s, R=PARAMS["R"], B=PARAMS["B"]): return 1 - (1 - np.asarray(s)**R)**B

# doc -> representative index, for candidate look-ups
doc2rep = np.empty(len(dev_ids), dtype=np.int64)
for r, g in enumerate(dd["members"]):
    doc2rep[g] = r
nr = dd["n_reps"]; cand_set = set(dd["cand"].tolist())
idx = {x:i for i,x in enumerate(dev_ids)}
def was_candidate(a, b):
    ra, rb = doc2rep[idx[a]], doc2rep[idx[b]]
    return ra == rb or (min(ra,rb)*nr + max(ra,rb)) in cand_set

print("\n--- MISSED pairs (in truth, not returned) ---")
missed = sorted(gold - dev_pairs, key=lambda x: -truth_J[x])
for a,b in missed[:max(3,len(missed))][:10]:
    J = exact_jaccard(dev_text[a], dev_text[b])
    print(f"{a} {b}  exact J={J:.4f}  truth J={truth_J[(a,b)]:.4f}  P(examined)={float(p_examined(J)):.4f}  was_candidate={was_candidate(a,b)}")
    print("   ->", "LSH miss: the pair was unlucky in all 21 bands (probability %.2f%% at this J)." % (100*(1-float(p_examined(J))))
          if not was_candidate(a,b) else "candidate but rejected -> investigate verification")
if not missed: print("none")

print("\n--- WRONG pairs (returned, not in truth) ---")
wrong = sorted(dev_pairs - gold)
for a,b in wrong[:10]:
    print(f"{a} {b}  exact J={exact_jaccard(dev_text[a], dev_text[b]):.4f}")
if not wrong:
    print("none: every returned pair is verified exactly, so false positives are impossible (barring a 2^-64 hash collision).")

# What would have been wrong WITHOUT verification: highest-J candidates rejected by the exact check
J = dd["cand_J"]; ii = dd["cand"] // nr; jj = dd["cand"] % nr
rej = np.argsort(-np.where((J >= 0) & (J < 0.8), J, -1))[:5]
print("\n--- Trap pairs LSH proposed but exact verification rejected (would be false alarms) ---")
for k in rej:
    if J[k] < 0: break
    a, b = dev_ids[dd["members"][ii[k]][0]], dev_ids[dd["members"][jj[k]][0]]
    print(f"{a} {b}  exact J={J[k]:.4f}  (< 0.8, correctly discarded)")

## 3. Complexity & scaling report
Notation: n docs, L average words/doc, N = nL total shingles, K = 126, B = 21, R = 6, C = #candidate pairs.

| Stage | Time | Space |
|---|---|---|
| 1 shingle hash + exact dedupe | O(N) (`np.unique` per doc: O(L log L) each → O(N log L)) | O(N) |
| 2 boilerplate df (`np.unique` over all shingles) | O(N log N) | O(N) |
| 3 MinHash | **O(N·K)** | O(nK) signatures, O(BLOCK·K) temp |
| 4 LSH banding (sort per band) | O(nK + nB log n) | O(n) per band, O(C) candidates |
| 5 exact verification | O(C·L) (sorted-array `searchsorted`, after an O(1) size filter) | O(1) extra |
| 6 expand groups | O(#output pairs) | O(#output pairs) |

Overall ≈ **O(N·K + N log N + C·L)** — near-linear in n when C = O(n) (near-duplicates are sparse); worst-case C = O(n²) only if many unrelated docs collide, which the boilerplate filter and size filter prevent.

In [ ]:
REPEATS = 2
SIZES = [5000, 10000, 20000, 40000, 80000]
scal = {}
stage_at = {}
for n in SIZES:
    ts = []
    for r in range(REPEATS):
        gc.collect(); t0 = time.perf_counter()
        _, dg = _find(ids_full[:n], texts_full[:n])
        ts.append(time.perf_counter() - t0)
        if r == 0: stage_at[n] = dg["times"]
    scal[n] = statistics.median(ts)
    print(f"n={n:6d}  median {scal[n]:7.1f}s   runs={[round(x,1) for x in ts]}  candidates={dg['n_candidates']}")

In [ ]:
def brute_force(ids, texts, t=0.8):
    S = [{hash(s) for s in shingle_set(x)} for x in texts]
    out = set(); n = len(S)
    for i in range(n):
        a = S[i]; la = len(a)
        for j in range(i+1, n):
            b = S[j]; inter = len(a & b)
            if inter and inter / (la + len(b) - inter) >= t:
                out.add(tuple(sorted((ids[i], ids[j]))))
    return out

BRUTE_SIZES = [500, 1000, 2000, 4000]      # 4000 takes a few minutes
brute = {}
for n in BRUTE_SIZES:
    t0 = time.perf_counter(); bp = brute_force(ids_full[:n], texts_full[:n]); brute[n] = time.perf_counter() - t0
    mp = find_similar_pairs(ids_full[:n], texts_full[:n])
    print(f"brute n={n}: {brute[n]:.1f}s | pairs brute={len(bp)} lsh={len(mp)} | agree={bp==mp} | missed={len(bp-mp)} extra={len(mp-bp)}")

In [ ]:
ns = np.array(SIZES, float); ts = np.array([scal[n] for n in SIZES])
slope, icpt = np.polyfit(np.log(ns), np.log(ts), 1)
bn = np.array(BRUTE_SIZES, float); bt = np.array([brute[n] for n in BRUTE_SIZES])
bslope, bicpt = np.polyfit(np.log(bn), np.log(bt), 1)
print(f"LSH pipeline empirical exponent: {slope:.3f}      brute-force empirical exponent: {bslope:.3f}")

plt.figure(figsize=(7,5))
plt.loglog(ns, ts, "o-", label=f"MinHash+LSH (slope {slope:.2f})")
plt.loglog(bn, bt, "s-", label=f"brute force (slope {bslope:.2f})")
xx = np.logspace(np.log10(500), np.log10(112571), 50)
plt.loglog(xx, np.exp(icpt)*xx**slope, "--", alpha=.4); plt.loglog(xx, np.exp(bicpt)*xx**bslope, "--", alpha=.4)
plt.axhline(900, color="r", ls=":", label="15-minute budget"); plt.xlabel("n documents"); plt.ylabel("seconds")
plt.grid(True, which="both", alpha=.3); plt.legend(); plt.title("Runtime scaling (log-log)"); plt.show()

t_full_pred = np.exp(bicpt) * 112571**bslope
n_imp = (900/np.exp(bicpt))**(1/bslope)
print(f"Brute force extrapolated to 112,571 docs: {t_full_pred/3600:.1f} hours;  exceeds 15 min at n ≈ {n_imp:,.0f}")
print(f"Brute-force pair count at 112,571 docs: {112571*112570//2:,.0f}")

print("\nStage breakdown at n=40,000 (first run):")
st = stage_at[40000]; tot = sum(st.values())
for k, v in st.items(): print(f"  {k:20s} {v:7.1f}s  {100*v/tot:5.1f}%")

### Written analysis (fill in the measured numbers from the cells above)
* **Stage complexities:** see the table at the top of section 3.
* **Bottleneck:** read it off the stage breakdown at n=40,000. Expected: MinHash (Θ(N·K) = 126 multiply-add-min operations per shingle) dominates, followed by shingle hashing (Python tokenising/hashing of every word). Banding and verification are tiny because C is small.
* **Empirical vs. theoretical slope:** theory is ≈1 (near-linear: every stage is linear in N except two sorts, giving only a log factor). The fitted exponent should land at roughly 1.0–1.15; the small excess comes from the `np.unique` sorts (N log N), cache effects as the arrays grow past CPU cache, and a slowly growing number of candidate pairs as more near-duplicate/boilerplate-sharing docs appear. If the slope is higher than expected, compare `n_candidates` across sizes.
* **Boilerplate:** thousands of articles share identical header/footer text. Naively this creates Θ(m²) candidate pairs for m boilerplate docs, since their sketches share many minima. Mitigation: (1) shingles with document frequency > cap (≈0.2% of docs, min 30) are removed *from the sketch only*, so the sketch describes the distinctive content; (2) a doc that would lose >50% of its shingles is sketched in full so its signature is not distorted; (3) verification always uses the *full* shingle set, so correctness of the 0.8 test is unaffected; (4) an exact size filter (J ≤ min/max) and a bucket-size valve bound worst-case work. See the ablation below.
* **Brute force:** Θ(n²·L) (n(n−1)/2 set intersections, each O(L)); the extrapolation cell above gives the n at which it passes the 15-minute budget on this machine, and the hours it would need for 112,571 docs (≈6.3·10⁹ pairs).

## 4. Parameter justification & probability curve

In [ ]:
# Choice of (B, R) for K ≈ 126: probability of examination at several similarities
rows = []
for B_, R_ in [(63,2),(42,3),(31,4),(25,5),(21,6),(18,7),(16,8),(14,9)]:
    f = lambda s: float(1-(1-s**R_)**B_)
    rows.append(dict(B=B_, R=R_, K=B_*R_, P_at_0_9=f(.9), P_at_0_8=f(.8), P_at_0_7=f(.7), P_at_0_5=f(.5), P_at_0_3=f(.3), P_at_0_1=f(.1)))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nChosen: K=126, B=21, R=6 -> P(J=0.8) = %.4f (misses ≈ %.2f%% of pairs exactly at threshold, less above)" % (float(p_examined(.8)), 100*(1-float(p_examined(.8)))))

In [ ]:
# Measured behaviour on dev: exact Jaccard of ALL pairs of distinct documents sharing ≥1 shingle (sparse matmul),
# then the fraction of them that LSH examined, per J-bin.
U_, offs_ = dd["U"], dd["offs"]
uniq, inv = np.unique(U_, return_inverse=True)
X = sparse.csr_matrix((np.ones(U_.size, dtype=np.int32), inv, offs_), shape=(nr, uniq.size))
G = sparse.triu(X @ X.T, k=1).tocoo()
lens_ = dd["lens"]
Jall = G.data / (lens_[G.row] + lens_[G.col] - G.data)
packed = G.row.astype(np.int64) * nr + G.col
is_cand = np.isin(packed, dd["cand"])

bins = np.arange(0, 1.0001, 0.05)
b_id = np.minimum((Jall / 0.05).astype(int), len(bins) - 2)
tot = np.bincount(b_id, minlength=len(bins)-1); hit = np.bincount(b_id, weights=is_cand, minlength=len(bins)-1)
ctr = (bins[:-1] + bins[1:]) / 2; frac = np.where(tot > 0, hit / np.maximum(tot, 1), np.nan)

s = np.linspace(0, 1, 500)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, logy in zip(ax, [False, True]):
    a.plot(s, p_examined(s), lw=2, label=f"theory 1-(1-J^{PARAMS['R']})^{PARAMS['B']}")
    m = tot >= 1
    a.scatter(ctr[m], frac[m], c="C1", s=np.clip(10 + np.log1p(tot[m])*8, 10, 90), label="measured (dev, per 0.05 bin)", zorder=3)
    a.axvline(0.8, color="r", ls="--", label="threshold 0.8"); a.set_xlabel("true Jaccard J"); a.set_ylabel("P(pair examined)")
    if logy: a.set_yscale("log"); a.set_ylim(1e-7, 1.5); a.set_title("log scale (tail: false-alarm load)")
    else: a.set_title("probability a pair is examined")
    a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()
print(pd.DataFrame(dict(J_bin_center=ctr, pairs=tot, examined=hit.astype(int), measured=frac.round(4), theory=p_examined(ctr).round(4))).to_string(index=False))

**Why K=126, R=6, B=21.**
* *Recall at the threshold.* P(examine) at J=0.8 is 0.998 (R=6) vs 0.95 for R=8 and ≈1 for R≤5. Pairs sit *just* below 0.8 on purpose and the key rewards recall ≥ 0.9 on pairs at/above 0.8 with edit noise, so a high P at 0.8 matters more than a razor-sharp step.
* *False-alarm load.* Candidates are verified exactly, so false alarms only cost time. R=6 gives P(0.5)=0.28, P(0.3)=0.015, P(0.1)≈2·10⁻⁵: unrelated or boilerplate-sharing pairs (J ≲ 0.2) almost never become candidates. R=2–4 would examine large fractions of boilerplate pairs.
* *Why not larger K.* MinHash cost is linear in K (the bottleneck), so K=126 keeps the full run to a few minutes; the measured candidate count and the dev recall show extra bands are unnecessary. If dev recall < 1 in a region of interest, raise B (e.g. K=168, B=28, R=6) at ~33% extra MinHash time.
* *Other settings:* `DF_CAP = max(30, 0.2% n)` — natural duplicates occur in groups of ≤ a handful of docs, boilerplate in thousands, so any value between ~10 and ~1000 separates them; `MAX_DROP=0.5` and `SHORT_M=20` protect short/boilerplate-heavy docs; `MAX_BUCKET=400` is only a safety valve (report `oversize_buckets`, expected 0).

In [ ]:
# Ablation on dev: effect of boilerplate filtering and of (B,R)
rows = []
for name, prm in [("default", {}), ("no df filter", dict(USE_DF=False)),
                  ("R=8,B=16 (K=128)", dict(K=128, R=8, B=16)), ("R=5,B=25 (K=125)", dict(K=125, R=5, B=25)),
                  ("K=168,B=28,R=6", dict(K=168, R=6, B=28))]:
    t0 = time.perf_counter(); pp, g = _find(dev_ids, dev_texts, params=prm); dt = time.perf_counter() - t0
    P_, R_, F_, _ = prf(pp, gold)
    rows.append(dict(config=name, seconds=round(dt,1), candidates=g["n_candidates"], precision=round(P_,4), recall=round(R_,4), F1=round(F_,4)))
pd.DataFrame(rows)